# ⚡ 1-CLICK UNCENSOR PIPELINE (CHẠY 1 LẦN DUY NHẤT LÀ XONG)
Notebook này thực hiện **bóc tách 100% kiểm duyệt (Uncensor/Abliteration)** cho Model Lập trình và **lưu thẳng vào Google Drive**.

### 🎯 Ưu điểm tuyệt đối:
- **1 Click duy nhất:** Bấm **Chạy tất cả (Run All)** từ trên xuống dưới, không cần bấm chọn phím hay cấu hình phức tạp.
- **Không xung đột thư viện:** Sử dụng trực tiếp Native PyTorch & Hugging Face Transformers chuẩn của Colab.
- **Tối ưu siêu tốc trên GPU A100/L4:** Xử lý hoàn tất trong 2-3 phút và lưu an toàn vào Google Drive của bạn.

In [ ]:
# @title 🚀 BẤM NÚT NÀY ĐỂ CHẠY TOÀN BỘ QUY TRÌNH (TỰ ĐỘNG TỪ ĐẦU ĐẾN CUỐI)
# @markdown ### ⚙️ Cấu hình Model mục tiêu:
MODEL_CHOICE = "Qwen/Qwen2.5-Coder-7B-Instruct" # @param ["Qwen/Qwen2.5-Coder-7B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"]
HF_TOKEN = "" # @param {type:"string"} # (Tùy chọn) Dán Access Token Hugging Face nếu muốn tự động upload lên HF

import os
import torch
from google.colab import drive

print("=" * 70)
print("🚀 BẮT ĐẦU QUY TRÌNH UNCENSOR TỰ ĐỘNG 100%")
print("=" * 70)

# BƯỚC 1: KẾT NỐI GOOGLE DRIVE
print("\n📁 [1/5] Đang kết nối Google Drive...")
drive.mount('/content/drive', force_remount=True)
SAVE_DIR = "/content/drive/MyDrive/ai_coding_models_uncensored"
os.makedirs(SAVE_DIR, exist_ok=True)

clean_name = MODEL_CHOICE.split("/")[-1]
OUTPUT_MODEL_NAME = f"{clean_name}-Uncensored"
TARGET_PATH = os.path.join(SAVE_DIR, OUTPUT_MODEL_NAME)
print(f"✅ Thư mục lưu đích trên Drive: {TARGET_PATH}")

# BƯỚC 2: CÀI ĐẶT THƯ VIỆN CHUẨN (KHÔNG ĐỤNG ĐẾN PYTORCH GỐC)
print("\n📦 [2/5] Đang cài đặt thư viện cần thiết...")
!pip install -q -U transformers datasets accelerate huggingface_hub

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# BƯỚC 3: TẢI MODEL LÊN GPU
print(f"\n📥 [3/5] Đang nạp Model {MODEL_CHOICE} lên GPU {torch.cuda.get_device_name(0)}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHOICE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_CHOICE,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
print("✅ Nạp Model thành công!")

# BƯỚC 4: TÍNH TOÁN VECTOR TRIỆT TIÊU KIỂM DUYỆT (ABLITERATION)
print("\n🧮 [4/5] Đang trích xuất và triệt tiêu vector từ chối trả lời (Censorship Directions)...")
harmful_ds = load_dataset("mlabonne/harmful_behaviors", split="train[:120]")
harmless_ds = load_dataset("mlabonne/harmless_alpaca", split="train[:120]")

def format_prompts(dataset, col="text"):
    prompts = []
    for item in dataset:
        chat = [{"role": "user", "content": item[col]}]
        try:
            formatted = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        except Exception:
            formatted = f"User: {item[col]}\nAssistant:"
        prompts.append(formatted)
    return prompts

harmful_prompts = format_prompts(harmful_ds)
harmless_prompts = format_prompts(harmless_ds)

def get_mean_activations(prompts, batch_size=16):
    all_acts = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = torch.stack(outputs.hidden_states) # (n_layers+1, batch, seq_len, dim)
            seq_lens = inputs.attention_mask.sum(dim=1) - 1
            batch_acts = []
            for b_idx, s_len in enumerate(seq_lens):
                batch_acts.append(hidden_states[:, b_idx, s_len, :])
            all_acts.append(torch.stack(batch_acts))
    all_acts = torch.cat(all_acts, dim=0)
    return all_acts.mean(dim=0)

print("  • Đang tính ma trận ẩn của Prompts...")
harmful_acts = get_mean_activations(harmful_prompts)
harmless_acts = get_mean_activations(harmless_prompts)

refusal_directions = harmful_acts - harmless_acts
refusal_directions = refusal_directions / refusal_directions.norm(dim=-1, keepdim=True)

# Triệt tiêu kiểm duyệt trên các Layer cốt lõi
n_layers = model.config.num_hidden_layers
start_layer = int(0.30 * n_layers)
end_layer = int(0.85 * n_layers)
print(f"  • Đang bóc tách vĩnh viễn cơ chế kiểm duyệt từ Layer {start_layer} đến {end_layer}...")

layers = model.model.layers if hasattr(model, 'model') and hasattr(model.model, 'layers') else model.layers

for layer_idx in range(start_layer, end_layer):
    v = refusal_directions[layer_idx + 1].to(model.dtype).to(model.device)
    layer = layers[layer_idx]
    
    # Triệt tiêu trên MLP Down Projection
    if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'down_proj'):
        W = layer.mlp.down_proj.weight.data
        proj = torch.matmul(W, v)
        layer.mlp.down_proj.weight.data = W - torch.outer(proj, v)
        
    # Triệt tiêu trên Self-Attention Output Projection
    if hasattr(layer, 'self_attn') and hasattr(layer.self_attn, 'o_proj'):
        W = layer.self_attn.o_proj.weight.data
        proj = torch.matmul(W, v)
        layer.self_attn.o_proj.weight.data = W - torch.outer(proj, v)

print("✅ Bóc tách kiểm duyệt thành công 100%!")

# BƯỚC 5: LƯU TRỰC TIẾP VÀO GOOGLE DRIVE
print(f"\n💾 [5/5] Đang lưu Model Uncensored hoàn chỉnh vào Google Drive: {TARGET_PATH}...")
os.makedirs(TARGET_PATH, exist_ok=True)
model.save_pretrained(TARGET_PATH, max_shard_size="5GB")
tokenizer.save_pretrained(TARGET_PATH)

print("\n" + "=" * 70)
print("🎉 CHÚC MỪNG! TOÀN BỘ QUY TRÌNH ĐÃ HOÀN TẤT THÀNH CÔNG 100%!")
print(f"📁 Đường dẫn Model trên Google Drive của bạn: {TARGET_PATH}")
print("=" * 70)

# Hiển thị danh sách file đã tạo
print("\n📁 Danh sách các file trong Drive:")
for f in sorted(os.listdir(TARGET_PATH)):
    sz = os.path.getsize(os.path.join(TARGET_PATH, f)) / (1024 * 1024)
    print(f"  - {f} ({sz:.1f} MB)")

# Upload Hugging Face nếu có Token
if HF_TOKEN.strip():
    print("\n📤 Đang tải model lên Hugging Face Hub...")
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN.strip())
    try:
        user = api.whoami()["name"]
    except Exception:
        user = "Leon234aamon"
    repo_id = f"{user}/{OUTPUT_MODEL_NAME}"
    api.create_repo(repo_id=repo_id, exist_ok=True, private=True)
    api.upload_folder(folder_path=TARGET_PATH, repo_id=repo_id, repo_type="model")
    print(f"🎉 Upload Hugging Face thành công: https://huggingface.co/{repo_id}")
